# ЛАБОРАТОРНА РОБОТА 5

In [78]:
import numpy as np
import random
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation

## Задача про рюкзак

Дано $N$ предметів, місткість рюкзака $W$, ваги предметів  
$w = \{w_1, w_2, \dots, w_N\}$ та вартості  
$p = \{p_1, p_2, \dots, p_N\}$.

Потрібно знайти набір бінарних змінних  
$B = \{b_1, b_2, \dots, b_N\}$, де:

- $b_i = 1$, якщо предмет $i$ включений у рюкзак
- $b_i = 0$, якщо предмет $i$ не включений

#### Обмеження:
$
\sum_{i=1}^{N} b_i w_i \leq W
$

#### Цільова функція:
$
\sum_{i=1}^{N} b_i p_i \rightarrow \max
$

## Генетичний алгоритм 

In [79]:
pop_size = 50
generations = 100

In [ ]:
def create_individual(n_items):

    individual = []
    for i in range(n_items):
        g = random.randint(0, 1) 
        individual.append(g)

    return individual

def fitness(ind, weights, values, W):
    
    total_weight = sum(ind[i] * weights[i] for i in range(len(ind)))
    total_value = sum(ind[i] * values[i] for i in range(len(ind)))

    if total_weight > W:
        return 0
    
    return total_value

def selection(population, weights, values, W):
    
    candidates = random.sample(population, 3)
    return max(candidates, key=lambda ind: fitness(ind, weights, values, W))

def crossover(parent1, parent2):
    
    size = len(parent1)
    child = []
    for i in range(size):
        
        if random.random() < 0.5:
            child.append(parent1[i]) 
        else:
            child.append(parent2[i])

    return child

def mutate(ind, mutation_prob=0.1):
    for i in range(len(ind)):
        if random.random() < mutation_prob:
            ind[i] = 1 - ind[i]   

In [ ]:
def ga(weights, values, W, mutation_prob=0.1):

    n_items = len(weights)

    population = [create_individual(n_items) for _ in range(pop_size)]

    best_ind = None
    best_value = 0

    best_history = []        
    population_history = []  

    for _ in range(generations):

        population.sort(key=lambda ind: fitness(ind, weights, values, W), reverse=True)

        current_best = population[0]
        current_value = fitness(current_best, weights, values, W)

        if current_value > best_value:
            best_value = current_value
            best_ind = current_best.copy()

        best_history.append(best_value)
        current_population_values = [
            fitness(ind, weights, values, W) for ind in population
        ]
        population_history.append(current_population_values)
        
        new_population = population[:1]

        while len(new_population) < pop_size:

            parent1 = selection(population, weights, values, W)
            parent2 = selection(population, weights, values, W)

            child = crossover(parent1, parent2)
            mutate(child, mutation_prob)

            new_population.append(child)

        population = new_population

    return best_ind, best_value, best_history, population_history

## Алгоритм імітації відпалу

In [ ]:
iterations = 50

In [ ]:
def energy(state, weights, values, W):
    
    total_w = sum(state[i] * weights[i] for i in range(len(state)))
    total_v = sum(state[i] * values[i] for i in range(len(state)))

    if total_w > W:
        return float("inf")
    return -total_v

def neighbor(state):
    new_state = state.copy()
    
    i, j = random.sample(range(len(state)), 2)
    new_state[i], new_state[j] = new_state[j], new_state[i]

    for k in range(len(new_state)):
        if random.random() < 0.1:   
            new_state[k] = 1 - new_state[k]

    return new_state

In [ ]:
def simulated_annealing(weights, values, W, t_max=1000, t_min=1, alpha=0.95):

    n = len(weights)
    
    current = [random.randint(0, 1) for _ in range(n)]
    current_energy = energy(current, weights, values, W)

    best = current.copy()
    best_energy = current_energy

    t = t_max

    history = []

    while t > t_min:

        for _ in range(iterations):

            candidate = neighbor(current)
            candidate_energy = energy(candidate, weights, values, W)

            delta_E = candidate_energy - current_energy

            if delta_E <= 0:
                current = candidate
                current_energy = candidate_energy

            else:
                p = math.exp(-delta_E / t)
                if random.random() < p:
                    current = candidate
                    current_energy = candidate_energy

            if current_energy < best_energy:
                best = current.copy()
                best_energy = current_energy

        t = t * alpha

        history.append(-best_energy) 

    return best, -best_energy, history

## Функція для візуалізації

In [ ]:
def create_gif(best_history, number, name, filename, history_values=None):

    if number == 1:
        fig, ax_conv = plt.subplots(1, 1, figsize=(8, 6))
        ax_hist = None
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        ax_hist, ax_conv = axes

    suptitle = fig.suptitle("", fontsize=14, fontweight="bold")

    ymin = np.min(best_history)
    ymax = np.max(best_history)

    frame_indices = list(range(0, len(best_history), 5))

    def update(i):
        frame_idx = frame_indices[i]  

        ax_conv.cla()
        if number == 2:
            ax_hist.cla()

        if number == 2 and history_values is not None:
            
            current_values = np.array(history_values[frame_idx])
            current_values = current_values[current_values > 0]

            ax_hist.hist(current_values, bins=20, color='skyblue', edgecolor='black')
            ax_hist.set_title("Гістограма")
            ax_hist.set_xlabel("Fitness")
            ax_hist.set_ylabel("Частота")
            ax_hist.grid()

        ax_conv.plot(best_history[:frame_idx + 1], lw=2, color='green')
        ax_conv.set_title("Збіжність")
        ax_conv.set_xlabel("Ітерація")
        ax_conv.set_ylabel("Найкраще значення")
        ax_conv.set_xlim(0, len(best_history))
        ax_conv.set_ylim(ymin, ymax)
        ax_conv.grid()

        suptitle.set_text(
            f"{name}\nІтерація {frame_idx} | Best: {best_history[frame_idx]:.2f}"
        )

        return (ax_hist, ax_conv) if number == 2 else (ax_conv,)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frame_indices), 
        interval = 500,
        repeat=False
    )

    ani.save(filename, writer="pillow")
    plt.close(fig)

## Генерація даних
- k - параметр, який визначає наскільки "великий" рюкзак відносно всіх предметів

In [ ]:
def generate_data(n_items, weight_range=(1, 50), value_range=(10, 100), k = 0.6):

    weights = np.random.randint(weight_range[0], weight_range[1] + 1, n_items)
    values = np.random.randint(value_range[0], value_range[1] + 1, n_items)

    W = int(k * sum(weights))

    return weights.tolist(), values.tolist(), W

## Результат

In [85]:
sizes = [30, 50, 80, 100]

for n in sizes:
    
    print(f"\n------ {n} предметів ------")

    weights, values, W = generate_data(n_items=n)

    ga_best, ga_value, ga_hist_best, ga_hist = ga(weights, values, W, mutation_prob=0.1)
    sa_best, sa_value, sa_hist_best = simulated_annealing(weights, values, W, t_max=1000, t_min=1, alpha=0.95)
    
    create_gif(ga_hist_best, 2, name = f"Genetic Algorithm n = {n}", filename = f"ga_{n}.gif", history_values = ga_hist)
    create_gif(sa_hist_best, 1, name = f"Simulated Annealing n = {n}", filename = f"sa_{n}.gif")

    ga_weight = sum(ga_best[i] * weights[i] for i in range(n))
    sa_weight = sum(sa_best[i] * weights[i] for i in range(n))

    print(f"\nGA результат:")
    print(f"Value: {ga_value:.2f}")
    print(f"Weight: {ga_weight} / {W}")
    print(f"Solution: {ga_best}")

    print(f"\nSA результат:")
    print(f"Value: {sa_value:.2f}")
    print(f"Weight: {sa_weight} / {W}")
    print(f"Solution: {sa_best}")


------ 30 предметів ------

GA результат:
Value: 1551.00
Weight: 441 / 444
Solution: [0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1]

SA результат:
Value: 1578.00
Weight: 434 / 444
Solution: [0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1]

------ 50 предметів ------

GA результат:
Value: 2144.00
Weight: 842 / 855
Solution: [1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1]

SA результат:
Value: 2241.00
Weight: 855 / 855
Solution: [1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1]

------ 80 предметів ------

GA результат:
Value: 2999.00
Weight: 1168 / 1191
Solution: [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0